# PoC step 2: validate and load
Runs the tests, then validates every record through the gateway and loads the canonical SQLite store.

**Before you start:** copy `gateway.py`, `loader.py`, and `test_gateway_loader.py` into `bw-interop-poc/src` in Drive (next to `schema.sql`, `participants.yaml`, `make_poc_pairs.py`). Step 1 must have produced `poc_pacs008.xml` and `poc_orange.json` in `data`.

In [1]:
# Cell 1: mount Drive, set paths, install dependencies
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/bw-interop-poc'
print('src :', sorted(os.listdir(f'{BASE}/src')))
print('data:', sorted(os.listdir(f'{BASE}/data')))
!pip -q install "pydantic>=2" pyyaml pytest

Mounted at /content/drive
src : ['gateway.py', 'loader.py', 'make_poc_pairs.py', 'participants.yaml', 'schema.sql', 'test_gateway_loader.py']
data: ['MatchedPoolReference.csv', 'poc_manifest.csv', 'poc_orange.json', 'poc_pacs008.xml']


In [2]:
# Cell 2: run the tests (expect: 36 passed)
!cd {BASE}/src && python -m pytest -q -p no:cacheprovider

....................................                                     [100%]
36 passed in 5.77s


In [3]:
# Cell 3: validate and load both sources into a fresh database
!python {BASE}/src/loader.py --reset --db {BASE}/data/poc.db --schema {BASE}/src/schema.sql --participants {BASE}/src/participants.yaml --xml {BASE}/data/poc_pacs008.xml --orange {BASE}/data/poc_orange.json

PACS008  rows_in= 402 mapped= 382 rejected= 20 reasons={'MISSING_CURRENCY': 6, 'DUPLICATE_KEY': 8, 'UNKNOWN_BIC': 6}
ORANGE   rows_in= 394 mapped= 394 rejected=  0 reasons={}
transactions             776
transaction_source_ids   3474
parties                  782
status_reports           776
exceptions               20
load_audit               2


In [4]:
# Cell 4: check the result (stops with an error if a count is wrong)
import sqlite3
import pandas as pd
db = sqlite3.connect(f'{BASE}/data/poc.db')
audit = pd.read_sql('SELECT batch_id, source_system, rows_in, rows_mapped, rows_rejected, reject_reasons FROM load_audit', db)
print(audit.to_string(index=False))
assert audit.set_index('source_system').loc['PACS008', ['rows_in', 'rows_mapped', 'rows_rejected']].tolist() == [402, 382, 20]
assert audit.set_index('source_system').loc['ORANGE', ['rows_in', 'rows_mapped', 'rows_rejected']].tolist() == [394, 394, 0]
counts = {t: db.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0] for t in ['transactions', 'transaction_source_ids', 'status_reports', 'exceptions']}
print(counts)
assert counts == {'transactions': 776, 'transaction_source_ids': 3474, 'status_reports': 776, 'exceptions': 20}
print()
print(pd.read_sql('SELECT reason_code, COUNT(*) AS n FROM exceptions GROUP BY reason_code', db).to_string(index=False))
pd.read_sql('SELECT * FROM v_transactions_flat LIMIT 5', db)

                    batch_id source_system  rows_in  rows_mapped  rows_rejected                                                reject_reasons
PACS008-2026-10-15T00:00:47Z       PACS008      402          382             20 {"MISSING_CURRENCY": 6, "DUPLICATE_KEY": 8, "UNKNOWN_BIC": 6}
 ORANGE-2026-10-15T00:00:48Z        ORANGE      394          394              0                                                            {}
{'transactions': 776, 'transaction_source_ids': 3474, 'status_reports': 776, 'exceptions': 20}

     reason_code  n
   DUPLICATE_KEY  8
MISSING_CURRENCY  6
     UNKNOWN_BIC  6


,txn_id,source_system,channel,direction,end_to_end_key,debtor_id_type,debtor_id,creditor_id_type,creditor_id,amount_minor,currency,charge_bearer,initiated_at,settled_at,status,status_source,batch_id
0,1,PACS008,BANK,PUSH,E2E-BW-20260807-0001,ACCOUNT,1155104816,MSISDN,26779399039,1852171,BWP,CRED,2026-10-15T08:00:00Z,2026-08-08,DISPATCHED,DERIVED_ACK,PACS008-2026-10-15T00:00:47Z
1,2,PACS008,BANK,PUSH,E2E-BW-20260830-0002,ACCOUNT,1703697384,MSISDN,26772119393,2525734,BWP,SHARED,2026-10-15T08:00:00Z,2026-09-02,DISPATCHED,DERIVED_ACK,PACS008-2026-10-15T00:00:47Z
2,3,PACS008,BANK,PUSH,E2E-BW-20260810-0003,ACCOUNT,2325609883,MSISDN,26771188152,2866104,BWP,CRED,2026-10-15T08:00:00Z,2026-08-10,DISPATCHED,DERIVED_ACK,PACS008-2026-10-15T00:00:47Z
3,4,PACS008,BANK,PUSH,E2E-BW-20260809-0005,ACCOUNT,9827557155,MSISDN,26771827415,553683,BWP,SHARED,2026-10-15T08:00:00Z,2026-08-11,DISPATCHED,DERIVED_ACK,PACS008-2026-10-15T00:00:47Z
4,5,PACS008,BANK,PUSH,E2E-BW-20260811-0006,ACCOUNT,2377816110,MSISDN,26771209814,4763856,BWP,SHARED,2026-10-15T08:00:00Z,2026-08-14,DISPATCHED,DERIVED_ACK,PACS008-2026-10-15T00:00:47Z


## Done when
- Cell 2 ends with `36 passed`.
- Cell 3 prints `PACS008 rows_in= 402 mapped= 382 rejected= 20` and `ORANGE rows_in= 394 mapped= 394 rejected= 0`.
- Cell 4 runs without an error and shows the reject reasons: MISSING_CURRENCY 6, UNKNOWN_BIC 6, DUPLICATE_KEY 8.

Paste the printed output of cells 2, 3, and 4 into the decision log.